# 10. Treinamento de Modelos - Pipeline Unificado Refatorado

## Filosofia MLOps Profissional

Este notebook consolida **TODA** a lógica de transformação e treinamento em um único fluxo reprodutível:

### Fluxo de Execução:
1. **Ingestão:** Carrega `df_treino` e `df_oot` (dados brutos processados)
2. **Feature Engineering Unificado:** 
   - Extração de features datetime (migrado de `09_transformacoes`)
   - Transformações numéricas (migrado de `07_transformacoes`)
   - Encoding categórico adaptativo (migrado de `09_transformacoes`)
3. **Pipeline sklearn.compose.ColumnTransformer:** 
   - Separação de pipelines por tipo de variável
   - Transformações aplicadas de forma consistente
4. **imblearn.pipeline.Pipeline:** 
   - Random Under Sampling (RUS) aplicado **APENAS** no `.fit()`
   - Zero Data Leakage
5. **Treinamento Multi-Modelo:** 5 algoritmos com hiperparâmetros ajustados
6. **Avaliação Treino + OOT:** Métricas realistas
7. **Persistência:** Pipeline completo salvo em `.pkl`

### Notebooks Antigos Substituídos:
- ❌ `07_transformacoes.ipynb` → Transformações numéricas agora no `ColumnTransformer`
- ❌ `09_transformacoes_das_variaveis.ipynb` → Feature engineering agora no Pipeline
- ❌ `10_treinamento_modelo.ipynb` → Versão antiga fragmentada

### Princípios:
- ✅ **Autossuficiência:** Não depende de notebooks anteriores
- ✅ **Reprodutibilidade:** Seeds fixas, paths dinâmicos (config.py)
- ✅ **Zero Hardcoding:** Usa `source.config` para todos os caminhos
- ✅ **Prevenção de Data Leakage:** Fit apenas no treino, transform em OOT
- ✅ **Metodologia Científica:** Validação temporal (OOT)

## 1. Setup - Importações e Configuração

In [31]:
# Configurar path para importar módulos do projeto
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))

# Importar configurações do projeto
from source.config import (
    PROJ_ROOT, 
    get_data_path, 
    get_model_path, 
    get_figure_path,
    ensure_directories
)

# Garantir que os diretórios existam
ensure_directories()

print(f"✅ Projeto Root: {PROJ_ROOT}")
print(f"✅ Configurações carregadas de source/config.py")

✅ Projeto Root: C:\Users\win\Desktop\TCC\money_laundering
✅ Configurações carregadas de source/config.py


In [32]:
# Importações padrão
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import json
import joblib

# Scikit-Learn - Preprocessing
from sklearn.preprocessing import (
    StandardScaler, 
    PowerTransformer,
    LabelEncoder,
    OneHotEncoder,
    FunctionTransformer
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# Scikit-Learn - Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier
)

# XGBoost e LightGBM
import xgboost as xgb
import lightgbm as lgb

# Imbalanced-Learn (RUS)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

# Category Encoders (Target Encoding)
from category_encoders import TargetEncoder

# Métricas
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

# Configurações
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Seed para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


## 2. Ingestão de Dados

Carrega datasets de treino e OOT previamente criados pelo notebook `04_divisao_treino_e_oot.ipynb`.

In [33]:
print("="*80)
print("CARREGAMENTO DOS DADOS")
print("="*80)

# Carregar datasets usando config.py
df_treino = pd.read_csv(get_data_path('df_treino.csv', 'processed'))
df_oot = pd.read_csv(get_data_path('df_oot.csv', 'processed'))

print(f"\n📊 Dataset de Treino:")
print(f"   Shape: {df_treino.shape}")
print(f"   Memória: {df_treino.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n📊 Dataset de OOT (Out-of-Time):")
print(f"   Shape: {df_oot.shape}")
print(f"   Memória: {df_oot.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n📋 Primeiras linhas do dataset de treino:")
display(df_treino.head())

print(f"\n📋 Tipos de dados:")
display(df_treino.dtypes.value_counts())

print(f"\n✅ Dados carregados com sucesso!")

CARREGAMENTO DOS DADOS

📊 Dataset de Treino:
   Shape: (25001186, 21)
   Memória: 24600.14 MB

📊 Dataset de OOT (Out-of-Time):
   Shape: (6250297, 21)
   Memória: 6148.21 MB

📋 Primeiras linhas do dataset de treino:


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From Bank Name,Bank ID,Account Number,From Entity ID,From Entity Name,To Bank Name,Bank ID_To,Account Number_To,To Entity ID,To Entity Name
0,2022-09-01,143137,835A3E1E0,143137,835A3E1E0,417.38,Mexican Peso,417.38,Mexican Peso,Reinvestment,0,Mexico Bank #50,143137,835A3E1E0,2AA22FC6990,Corporation #126451,Mexico Bank #50,143137,835A3E1E0,2AA22FC6990,Corporation #126451
1,2022-09-01,362692,81C920DA0,362692,81C920DA0,10370.35,US Dollar,10370.35,US Dollar,Reinvestment,0,Flagstone Trust Bank,362692,81C920DA0,2AA234A1D80,Partnership #10270,Flagstone Trust Bank,362692,81C920DA0,2AA234A1D80,Partnership #10270
2,2022-09-01,2120929,84346D460,2120929,84346D460,386382.55,US Dollar,386382.55,US Dollar,Reinvestment,0,National Bank of Newbury,2120929,84346D460,2AA20F1E430,Corporation #53588,National Bank of Newbury,2120929,84346D460,2AA20F1E430,Corporation #53588
3,2022-09-01,67054,81C91D960,67054,81C91D960,1955198.25,US Dollar,1955198.25,US Dollar,Reinvestment,0,First Bank of Los Angeles,67054,81C91D960,2AA20AA72E0,Partnership #31123,First Bank of Los Angeles,67054,81C91D960,2AA20AA72E0,Partnership #31123
4,2022-09-01,175944,841243680,175944,841243680,198525.25,Swiss Franc,198525.25,Swiss Franc,Reinvestment,0,Switzerland Bank #16,175944,841243680,2AA233762A0,Sole Proprietorship #138109,Switzerland Bank #16,175944,841243680,2AA233762A0,Sole Proprietorship #138109



📋 Tipos de dados:


object     14
int64       5
float64     2
Name: count, dtype: int64


✅ Dados carregados com sucesso!


## 3. Identificação da Target e Features

Identifica automaticamente a variável target (binária) e separa features.

In [34]:
# Possíveis nomes de target
possible_targets = ['is_laundering', 'Is Laundering', 'flag', 'target', 'label', 'is_fraud']
target_col = None

# Procurar target
for col in possible_targets:
    if col in df_treino.columns:
        target_col = col
        break

# Se não encontrar, procurar coluna binária
if target_col is None:
    for col in df_treino.columns:
        if df_treino[col].nunique() == 2 and set(df_treino[col].unique()).issubset({0, 1, '0', '1'}):
            target_col = col
            print(f"⚠️ Target identificado automaticamente: {col}")
            break

if target_col is None:
    raise ValueError("❌ Target não encontrado! Verifique os dados.")

# Separar features e target
X_train = df_treino.drop(columns=[target_col])
y_train = df_treino[target_col].astype(int)

X_oot = df_oot.drop(columns=[target_col])
y_oot = df_oot[target_col].astype(int)

print("="*80)
print("SEPARAÇÃO DE FEATURES E TARGET")
print("="*80)
print(f"\n🎯 Variável Target: {target_col}")
print(f"\n📊 Distribuição da Target (Treino):")
print(y_train.value_counts())
print(f"\n   Taxa de Positivos: {y_train.mean():.2%}")

print(f"\n📊 Distribuição da Target (OOT):")
print(y_oot.value_counts())
print(f"\n   Taxa de Positivos: {y_oot.mean():.2%}")

print(f"\n✅ Separação concluída!")
print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   X_oot: {X_oot.shape}")
print(f"   y_oot: {y_oot.shape}")

SEPARAÇÃO DE FEATURES E TARGET

🎯 Variável Target: Is Laundering

📊 Distribuição da Target (Treino):
Is Laundering
0    24988928
1       12258
Name: count, dtype: int64

   Taxa de Positivos: 0.05%

📊 Distribuição da Target (OOT):
Is Laundering
0    6246514
1       3783
Name: count, dtype: int64

   Taxa de Positivos: 0.06%

✅ Separação concluída!
   X_train: (25001186, 20)
   y_train: (25001186,)
   X_oot: (6250297, 20)
   y_oot: (6250297,)


## 4. Classificação Automática de Variáveis

Identifica tipos de variáveis para aplicar transformações apropriadas:
- **Numéricas:** Transformações matemáticas + escala
- **Categóricas:** Encoding adaptativo (OHE, Target, Frequency)
- **Datetime:** Extração de features temporais

In [35]:
# Identificar tipos de variáveis
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
datetime_cols = X_train.select_dtypes(include=['datetime64']).columns.tolist()

# Tentar converter colunas que podem ser datetime
for col in categorical_cols.copy():
    if 'date' in col.lower() or 'time' in col.lower() or 'timestamp' in col.lower():
        try:
            X_train[col] = pd.to_datetime(X_train[col], errors='coerce')
            X_oot[col] = pd.to_datetime(X_oot[col], errors='coerce')
            datetime_cols.append(col)
            categorical_cols.remove(col)
            print(f"✅ Coluna '{col}' convertida para datetime")
        except:
            pass

# Calcular cardinalidade das categóricas
cardinality = {}
for col in categorical_cols:
    cardinality[col] = X_train[col].nunique()

# Separar por estratégia de encoding
binary_vars = [col for col, n in cardinality.items() if n == 2]
onehot_vars = [col for col, n in cardinality.items() if 2 < n <= 10]
target_vars = [col for col, n in cardinality.items() if 10 < n <= 50]
frequency_vars = [col for col, n in cardinality.items() if n > 50]

print("="*80)
print("CLASSIFICAÇÃO AUTOMÁTICA DE VARIÁVEIS")
print("="*80)
print(f"\n📊 Variáveis Numéricas: {len(numeric_cols)}")
if len(numeric_cols) > 0 and len(numeric_cols) <= 20:
    print(f"   {numeric_cols}")

print(f"\n📝 Variáveis Categóricas: {len(categorical_cols)}")
print(f"   - Binárias (Label Encoding): {len(binary_vars)}")
if binary_vars:
    print(f"     {binary_vars}")
print(f"   - Baixa Cardinalidade (One-Hot): {len(onehot_vars)}")
if onehot_vars:
    print(f"     {onehot_vars}")
print(f"   - Média Cardinalidade (Target Encoding): {len(target_vars)}")
if target_vars:
    print(f"     {target_vars}")
print(f"   - Alta Cardinalidade (Frequency): {len(frequency_vars)}")
if frequency_vars:
    print(f"     {frequency_vars[:5]}..." if len(frequency_vars) > 5 else f"     {frequency_vars}")

print(f"\n📅 Variáveis Datetime: {len(datetime_cols)}")
if datetime_cols:
    print(f"   {datetime_cols}")

✅ Coluna 'Timestamp' convertida para datetime
CLASSIFICAÇÃO AUTOMÁTICA DE VARIÁVEIS

📊 Variáveis Numéricas: 6
   ['From Bank', 'To Bank', 'Amount Received', 'Amount Paid', 'Bank ID', 'Bank ID_To']

📝 Variáveis Categóricas: 13
   - Binárias (Label Encoding): 0
   - Baixa Cardinalidade (One-Hot): 1
     ['Payment Format']
   - Média Cardinalidade (Target Encoding): 2
     ['Receiving Currency', 'Payment Currency']
   - Alta Cardinalidade (Frequency): 10
     ['From Account', 'To Account', 'From Bank Name', 'Account Number', 'From Entity ID']...

📅 Variáveis Datetime: 1
   ['Timestamp']


## 5. Feature Engineering - Datetime Features

**Migrado de `09_transformacoes_das_variaveis.ipynb`**

Extrai 19 features de cada variável datetime:
- Componentes temporais: year, month, day, hour, etc.
- Features cíclicas: sin/cos para capturar periodicidade
- Features de negócio: weekend, business hours, night

In [ ]:
class DatetimeFeatureExtractor(BaseEstimator, TransformerMixin):
    """Extrai features de variáveis datetime (MIGRADO do notebook 09)"""
    
    def __init__(self, datetime_cols):
        self.datetime_cols = datetime_cols
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        for col in self.datetime_cols:
            if col in X_copy.columns:
                # Garantir que é datetime
                if X_copy[col].dtype != 'datetime64[ns]':
                    X_copy[col] = pd.to_datetime(X_copy[col], errors='coerce')
                
                # Componentes temporais (fillna para lidar com NaT)
                X_copy[f'{col}_year'] = X_copy[col].dt.year.fillna(0).astype(int)
                X_copy[f'{col}_month'] = X_copy[col].dt.month.fillna(0).astype(int)
                X_copy[f'{col}_day'] = X_copy[col].dt.day.fillna(0).astype(int)
                X_copy[f'{col}_dayofweek'] = X_copy[col].dt.dayofweek.fillna(0).astype(int)
                X_copy[f'{col}_hour'] = X_copy[col].dt.hour.fillna(0).astype(int)
                X_copy[f'{col}_minute'] = X_copy[col].dt.minute.fillna(0).astype(int)
                X_copy[f'{col}_quarter'] = X_copy[col].dt.quarter.fillna(0).astype(int)
                X_copy[f'{col}_dayofyear'] = X_copy[col].dt.dayofyear.fillna(0).astype(int)
                X_copy[f'{col}_weekofyear'] = X_copy[col].dt.isocalendar().week.fillna(0).astype(int)
                
                # Features cíclicas (para capturar periodicidade)
                X_copy[f'{col}_month_sin'] = np.sin(2 * np.pi * X_copy[f'{col}_month'] / 12)
                X_copy[f'{col}_month_cos'] = np.cos(2 * np.pi * X_copy[f'{col}_month'] / 12)
                X_copy[f'{col}_dayofweek_sin'] = np.sin(2 * np.pi * X_copy[f'{col}_dayofweek'] / 7)
                X_copy[f'{col}_dayofweek_cos'] = np.cos(2 * np.pi * X_copy[f'{col}_dayofweek'] / 7)
                X_copy[f'{col}_hour_sin'] = np.sin(2 * np.pi * X_copy[f'{col}_hour'] / 24)
                X_copy[f'{col}_hour_cos'] = np.cos(2 * np.pi * X_copy[f'{col}_hour'] / 24)
                
                # Features de negócio
                X_copy[f'{col}_is_weekend'] = X_copy[f'{col}_dayofweek'].isin([5, 6]).astype(int)
                X_copy[f'{col}_is_business_hours'] = X_copy[f'{col}_hour'].between(9, 17).astype(int)
                X_copy[f'{col}_is_night'] = (X_copy[f'{col}_hour'].between(22, 23) | X_copy[f'{col}_hour'].between(0, 5)).astype(int)
                
                # Remover coluna datetime original (não é mais necessária)
                X_copy = X_copy.drop(columns=[col])
        
        return X_copy

# Aplicar extração de features datetime se houver
if len(datetime_cols) > 0:
    print("Aplicando extração de features datetime...\n")
    datetime_extractor = DatetimeFeatureExtractor(datetime_cols)
    
    X_train = datetime_extractor.fit_transform(X_train)
    X_oot = datetime_extractor.transform(X_oot)
    
    print(f"✅ Features extraídas de {len(datetime_cols)} variáveis datetime")
    print(f"   {len(datetime_cols) * 19} novas features criadas")
    print(f"   ✅ Coluna datetime original removida (não mais necessária)")
    print(f"\n   Novo shape:")
    print(f"   - X_train: {X_train.shape}")
    print(f"   - X_oot: {X_oot.shape}")
    
    # Atualizar lista de colunas numéricas (excluindo datetime da lista)
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
else:
    print("⚠️ Nenhuma variável datetime encontrada. Pulando extração de features temporais.")

Aplicando extração de features datetime...

✅ Features extraídas de 1 variáveis datetime
   19 novas features criadas
   ⚠️ Coluna datetime original mantida para compatibilidade com pipeline

   Novo shape:
   - X_train: (25001186, 37)
   - X_oot: (6250297, 37)


## 6. Transformadores Customizados

**Migrado de `07_transformacoes.ipynb`**

Transformações matemáticas para corrigir distribuições assimétricas.

In [37]:
class LogTransformer(BaseEstimator, TransformerMixin):
    """Aplica log(1+x) para evitar log(0) (MIGRADO do notebook 07)"""
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return np.log1p(np.abs(X))  # log(1 + |x|) para lidar com negativos


class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Frequency Encoding para variáveis de alta cardinalidade (MIGRADO do notebook 09)"""
    
    def __init__(self):
        self.freq_maps = {}
    
    def fit(self, X, y=None):
        # X é um DataFrame com colunas categóricas
        for col in X.columns:
            freq = X[col].value_counts(normalize=True).to_dict()
            self.freq_maps[col] = freq
        return self
    
    def transform(self, X):
        X_transformed = X.copy()
        for col in X.columns:
            # Mapear frequências, usar 0 para valores desconhecidos
            X_transformed[col] = X[col].map(self.freq_maps[col]).fillna(0)
        return X_transformed


print("✅ Transformadores customizados definidos:")
print("   - LogTransformer (log1p)")
print("   - FrequencyEncoder")

✅ Transformadores customizados definidos:
   - LogTransformer (log1p)
   - FrequencyEncoder


## 7. Criação do ColumnTransformer

Pipeline unificado que aplica transformações apropriadas para cada tipo de variável:
- **Numéricas:** Log → StandardScaler
- **Categóricas Binárias:** LabelEncoder
- **Categóricas Baixa Cardinalidade:** OneHotEncoder
- **Categóricas Média Cardinalidade:** TargetEncoder
- **Categóricas Alta Cardinalidade:** FrequencyEncoder

In [38]:
# Lista de transformadores
transformers_list = []

# 1. Pipeline para variáveis numéricas
if len(numeric_cols) > 0:
    numeric_pipeline = Pipeline([
        ('log', LogTransformer()),
        ('scaler', StandardScaler())
    ])
    transformers_list.append(('numeric', numeric_pipeline, numeric_cols))
    print(f"✅ Pipeline numérico criado: {len(numeric_cols)} variáveis")

# 2. Pipeline para variáveis categóricas - One-Hot Encoding
if len(onehot_vars) > 0:
    onehot_pipeline = Pipeline([
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    transformers_list.append(('onehot', onehot_pipeline, onehot_vars))
    print(f"✅ Pipeline One-Hot criado: {len(onehot_vars)} variáveis")

# 3. Pipeline para variáveis categóricas - Target Encoding
if len(target_vars) > 0:
    target_pipeline = Pipeline([
        ('target_enc', TargetEncoder(smoothing=1.0, min_samples_leaf=1))
    ])
    transformers_list.append(('target', target_pipeline, target_vars))
    print(f"✅ Pipeline Target Encoding criado: {len(target_vars)} variáveis")

# 4. Pipeline para variáveis categóricas - Frequency Encoding
if len(frequency_vars) > 0:
    frequency_pipeline = Pipeline([
        ('freq_enc', FrequencyEncoder())
    ])
    transformers_list.append(('frequency', frequency_pipeline, frequency_vars))
    print(f"✅ Pipeline Frequency Encoding criado: {len(frequency_vars)} variáveis")

# 5. Label Encoding para variáveis binárias (aplicado manualmente antes do ColumnTransformer)
if len(binary_vars) > 0:
    print(f"\n⚠️ Aplicando Label Encoding em {len(binary_vars)} variáveis binárias...")
    label_encoders = {}
    
    for var in binary_vars:
        le = LabelEncoder()
        # Fit no treino
        X_train[f'{var}_encoded'] = le.fit_transform(X_train[var].astype(str))
        label_encoders[var] = le
        
        # Transform no OOT (tratar valores desconhecidos)
        oot_values = X_oot[var].astype(str)
        oot_encoded = []
        for val in oot_values:
            if val in le.classes_:
                oot_encoded.append(le.transform([val])[0])
            else:
                oot_encoded.append(-1)  # Valor desconhecido
        X_oot[f'{var}_encoded'] = oot_encoded
        
        print(f"   ✅ {var}: {list(le.classes_)} → {list(range(len(le.classes_)))}")
    
    # Adicionar colunas encodadas às numéricas e remover originais
    encoded_cols = [f'{var}_encoded' for var in binary_vars]
    numeric_cols.extend(encoded_cols)
    
    X_train = X_train.drop(columns=binary_vars)
    X_oot = X_oot.drop(columns=binary_vars)
    
    # Remover de categóricas
    categorical_cols = [col for col in categorical_cols if col not in binary_vars]
    
    # Atualizar transformers_list
    transformers_list = [(name, pipe, cols) for name, pipe, cols in transformers_list if name != 'numeric']
    if len(numeric_cols) > 0:
        numeric_pipeline = Pipeline([
            ('log', LogTransformer()),
            ('scaler', StandardScaler())
        ])
        transformers_list.insert(0, ('numeric', numeric_pipeline, numeric_cols))

# Criar ColumnTransformer
if len(transformers_list) > 0:
    preprocessor = ColumnTransformer(
        transformers=transformers_list,
        remainder='drop',  # Drop colunas não especificadas
        verbose_feature_names_out=False
    )
    
    print("\n" + "="*80)
    print("COLUMN TRANSFORMER CRIADO COM SUCESSO")
    print("="*80)
    print(f"\nTotal de pipelines: {len(transformers_list)}")
    for name, _, cols in transformers_list:
        print(f"   - {name}: {len(cols)} variáveis")
else:
    raise ValueError("❌ Nenhum transformador foi criado. Verifique os dados.")

✅ Pipeline numérico criado: 24 variáveis
✅ Pipeline One-Hot criado: 1 variáveis
✅ Pipeline Target Encoding criado: 2 variáveis
✅ Pipeline Frequency Encoding criado: 10 variáveis

COLUMN TRANSFORMER CRIADO COM SUCESSO

Total de pipelines: 4
   - numeric: 24 variáveis
   - onehot: 1 variáveis
   - target: 2 variáveis
   - frequency: 10 variáveis


## 8. Criação do Pipeline Completo com RUS

Integra:
1. **Preprocessamento** (ColumnTransformer)
2. **Random Under Sampling** (apenas no fit)
3. **Modelo de ML**

In [39]:
def create_full_pipeline(model, use_rus=True):
    """
    Cria pipeline completo: Preprocessor → RUS → Model
    
    Args:
        model: Estimador sklearn
        use_rus: Se True, aplica Random Under Sampling
    
    Returns:
        ImbPipeline se use_rus=True, senão Pipeline normal
    """
    steps = [('preprocessor', preprocessor)]
    
    if use_rus:
        steps.append(('rus', RandomUnderSampler(random_state=RANDOM_STATE)))
    
    steps.append(('classifier', model))
    
    if use_rus:
        return ImbPipeline(steps)
    else:
        return Pipeline(steps)


# Dicionário de modelos
models = {
    'Logistic Regression': LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        class_weight='balanced',
        solver='liblinear'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=4,
        class_weight='balanced',
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        use_label_encoder=False
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        verbose=-1
    )
}

# Criar pipelines
pipelines = {}
for name, model in models.items():
    pipelines[name] = create_full_pipeline(model, use_rus=True)

print("="*80)
print("PIPELINES COMPLETOS CRIADOS")
print("="*80)
print(f"\nTotal: {len(pipelines)} pipelines")
print(f"\nModelos:")
for i, name in enumerate(pipelines.keys(), 1):
    print(f"   {i}. {name}")

print(f"\n✅ Cada pipeline contém:")
print(f"   1. ColumnTransformer (feature engineering)")
print(f"   2. RandomUnderSampler (balanceamento no fit)")
print(f"   3. Modelo de ML")

PIPELINES COMPLETOS CRIADOS

Total: 5 pipelines

Modelos:
   1. Logistic Regression
   2. Random Forest
   3. Gradient Boosting
   4. XGBoost
   5. LightGBM

✅ Cada pipeline contém:
   1. ColumnTransformer (feature engineering)
   2. RandomUnderSampler (balanceamento no fit)
   3. Modelo de ML


## 9. Treinamento dos Modelos

Treina todos os pipelines nos dados de treino.

In [ ]:
print("="*80)
print("TREINAMENTO DOS MODELOS")
print("="*80)

trained_models = {}
training_times = {}

for name, pipeline in pipelines.items():
    print(f"\n🔄 Treinando: {name}...")
    start_time = datetime.now()
    
    try:
        # Treinar pipeline completo
        pipeline.fit(X_train, y_train)
        
        # Calcular tempo
        training_time = (datetime.now() - start_time).total_seconds()
        
        # Armazenar
        trained_models[name] = pipeline
        training_times[name] = training_time
        
        print(f"   ✅ Concluído em {training_time:.2f}s")
        
    except Exception as e:
        print(f"   ❌ Erro: {str(e)}")

print("\n" + "="*80)
print("TREINAMENTO CONCLUÍDO")
print("="*80)
print(f"\nModelos treinados: {len(trained_models)}")
print(f"\n📊 Tempos de Treinamento:")
for name, time in sorted(training_times.items(), key=lambda x: x[1]):
    print(f"   {name}: {time:.2f}s")

TREINAMENTO DOS MODELOS

🔄 Treinando: Logistic Regression...
   ❌ Erro: Unable to allocate 4.47 GiB for an array with shape (25001186, 24) and data type float64

🔄 Treinando: Random Forest...


## 10. Avaliação dos Modelos

Calcula métricas em **treino** e **OOT** para detectar overfitting.

In [28]:
# Avaliar todos os modelos - VERSÃO SIMPLIFICADA (economia de memória)
print("="*80)
print("AVALIAÇÃO DOS MODELOS (Amostra 50k registros)")
print("="*80)

# Amostrar dados para avaliação
np.random.seed(RANDOM_STATE)
train_sample_idx = np.random.choice(len(X_train), min(50000, len(X_train)), replace=False)
oot_sample_idx = np.random.choice(len(X_oot), min(50000, len(X_oot)), replace=False)

X_train_sample = X_train.iloc[train_sample_idx].copy()
y_train_sample = y_train.iloc[train_sample_idx].copy()
X_oot_sample = X_oot.iloc[oot_sample_idx].copy()
y_oot_sample = y_oot.iloc[oot_sample_idx].copy()

print(f"\n📊 Amostras criadas:")
print(f"   Train: {X_train_sample.shape[0]:,} registros")
print(f"   OOT: {X_oot_sample.shape[0]:,} registros")

results = []

for name in trained_models.keys():
    print(f"\n🔍 Avaliando: {name}")
    model = trained_models[name]
    
    try:
        # Predições no treino
        y_train_pred = model.predict(X_train_sample)
        y_train_proba = model.predict_proba(X_train_sample)[:, 1]
        
        # Métricas treino
        train_acc = accuracy_score(y_train_sample, y_train_pred)
        train_roc = roc_auc_score(y_train_sample, y_train_proba)
        
        results.append({
            'Model': name,
            'Dataset': 'Train',
            'Accuracy': train_acc,
            'Precision': precision_score(y_train_sample, y_train_pred, zero_division=0),
            'Recall': recall_score(y_train_sample, y_train_pred, zero_division=0),
            'F1-Score': f1_score(y_train_sample, y_train_pred, zero_division=0),
            'ROC-AUC': train_roc,
            'Avg Precision': average_precision_score(y_train_sample, y_train_proba)
        })
        
        # Predições no OOT
        y_oot_pred = model.predict(X_oot_sample)
        y_oot_proba = model.predict_proba(X_oot_sample)[:, 1]
        
        # Métricas OOT
        oot_acc = accuracy_score(y_oot_sample, y_oot_pred)
        oot_roc = roc_auc_score(y_oot_sample, y_oot_proba)
        
        results.append({
            'Model': name,
            'Dataset': 'OOT',
            'Accuracy': oot_acc,
            'Precision': precision_score(y_oot_sample, y_oot_pred, zero_division=0),
            'Recall': recall_score(y_oot_sample, y_oot_pred, zero_division=0),
            'F1-Score': f1_score(y_oot_sample, y_oot_pred, zero_division=0),
            'ROC-AUC': oot_roc,
            'Avg Precision': average_precision_score(y_oot_sample, y_oot_proba)
        })
        
        print(f"   ✅ Train: Acc={train_acc:.4f}, ROC-AUC={train_roc:.4f}")
        print(f"   ✅ OOT:   Acc={oot_acc:.4f}, ROC-AUC={oot_roc:.4f}")
        
    except Exception as e:
        print(f"   ❌ Erro: {str(e)[:100]}")

# Criar DataFrame
if len(results) > 0:
    df_results = pd.DataFrame(results)
    
    print("\n" + "="*80)
    print("RESULTADOS CONSOLIDADOS")
    print("="*80)
    display(df_results)
    
    # Salvar
    results_path = get_data_path('model_results_refatorado.csv', 'processed')
    df_results.to_csv(results_path, index=False)
    print(f"\n✅ Resultados salvos em: {results_path}")
else:
    print("\n❌ Nenhum modelo avaliado com sucesso.")

AVALIAÇÃO DOS MODELOS (Amostra 50k registros)

📊 Amostras criadas:
   Train: 50,000 registros
   OOT: 50,000 registros

🔍 Avaliando: Logistic Regression
   ❌ Erro: Pipeline is not fitted yet.

🔍 Avaliando: Random Forest
   ❌ Erro: Pipeline is not fitted yet.

🔍 Avaliando: Gradient Boosting
   ❌ Erro: Pipeline is not fitted yet.

🔍 Avaliando: XGBoost
   ❌ Erro: Pipeline is not fitted yet.

🔍 Avaliando: LightGBM
   ❌ Erro: Pipeline is not fitted yet.

❌ Nenhum modelo avaliado com sucesso.


In [30]:
# Debug: verificar colunas e preprocessor
print(f"X_train shape: {X_train.shape}")
print(f"X_train columns ({len(X_train.columns)}): {sorted(X_train.columns.tolist())[:10]}...")

print(f"\npreprocessor transformers:")
for name, transformer, cols in preprocessor.transformers:
    print(f"  {name}: {len(cols)} colunas")
    print(f"    Primeiras 3: {cols[:3]}")
    
print(f"\n\n✅ Verificando se colunas do preprocessor existem em X_train:")
all_cols_needed = []
for name, transformer, cols in preprocessor.transformers:
    all_cols_needed.extend(cols)
    
missing = [col for col in all_cols_needed if col not in X_train.columns]
if len(missing) > 0:
    print(f"  ❌ Colunas FALTANDO: {missing}")
else:
    print(f"  ✅ Todas as {len(all_cols_needed)} colunas existem!")
    
# Verificar colunas extras
extra_cols = [col for col in X_train.columns if col not in all_cols_needed]
if len(extra_cols) > 0:
    print(f"\n  ⚠️ Colunas EXTRAS em X_train (não usadas pelo preprocessor): {len(extra_cols)}")
    print(f"     {extra_cols}")

X_train shape: (25001186, 38)
X_train columns (38): ['Account Number', 'Account Number_To', 'Amount Paid', 'Amount Received', 'Bank ID', 'Bank ID_To', 'From Account', 'From Bank', 'From Bank Name', 'From Entity ID']...

preprocessor transformers:
  numeric: 24 colunas
    Primeiras 3: ['From Bank', 'To Bank', 'Amount Received']
  onehot: 1 colunas
    Primeiras 3: ['Payment Format']
  target: 2 colunas
    Primeiras 3: ['Receiving Currency', 'Payment Currency']
  frequency: 10 colunas
    Primeiras 3: ['From Account', 'To Account', 'From Bank Name']


✅ Verificando se colunas do preprocessor existem em X_train:
  ✅ Todas as 37 colunas existem!

  ⚠️ Colunas EXTRAS em X_train (não usadas pelo preprocessor): 1
     ['Timestamp']


## 11. Comparação Visual de Modelos

In [ ]:
# Comparar métricas no OOT
df_oot_results = df_results[df_results['Dataset'] == 'OOT'].copy()
df_oot_results = df_oot_results.set_index('Model')

# Plotar
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Comparação de Modelos - Dataset OOT', fontsize=16, fontweight='bold')

metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Avg Precision']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    df_oot_results[metric].plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(metric, fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_xlim([0, 1])
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(get_figure_path('model_comparison_refatorado.png'), dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Gráfico salvo!")

## 12. Seleção do Melhor Modelo e Persistência

Salva o melhor modelo (baseado em ROC-AUC no OOT).

In [ ]:
# Identificar melhor modelo (ROC-AUC no OOT)
best_model_name = df_oot_results['ROC-AUC'].idxmax()
best_model = trained_models[best_model_name]
best_roc_auc = df_oot_results.loc[best_model_name, 'ROC-AUC']

print("="*80)
print("MELHOR MODELO IDENTIFICADO")
print("="*80)
print(f"\n🏆 Modelo: {best_model_name}")
print(f"📊 ROC-AUC (OOT): {best_roc_auc:.4f}")
print(f"\nMétricas completas (OOT):")
print(df_oot_results.loc[best_model_name])

# Salvar pipeline completo
model_path = get_model_path(f'best_model_refatorado_{best_model_name.lower().replace(" ", "_")}.pkl')
joblib.dump(best_model, model_path)
print(f"\n✅ Pipeline completo salvo em: {model_path}")

# Salvar metadados
metadata = {
    'model_name': best_model_name,
    'training_date': datetime.now().isoformat(),
    'random_state': RANDOM_STATE,
    'metrics_oot': df_oot_results.loc[best_model_name].to_dict(),
    'training_time_seconds': training_times[best_model_name],
    'features_count': X_train.shape[1],
    'train_samples': X_train.shape[0],
    'oot_samples': X_oot.shape[0],
    'target_column': target_col,
    'transformations': {
        'numeric_cols': len(numeric_cols),
        'onehot_cols': len(onehot_vars),
        'target_encoding_cols': len(target_vars),
        'frequency_cols': len(frequency_vars),
        'datetime_cols': len(datetime_cols)
    }
}

metadata_path = get_model_path('training_info_refatorado.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadados salvos em: {metadata_path}")

## 13. Exemplo de Inferência

Demonstra como carregar o pipeline e fazer predições em novos dados.

In [ ]:
print("="*80)
print("EXEMPLO DE INFERÊNCIA")
print("="*80)

# Carregar pipeline salvo
loaded_pipeline = joblib.load(model_path)
print(f"\n✅ Pipeline carregado de: {model_path}")

# Pegar amostra de 5 registros do OOT
sample_data = X_oot.head(5)
sample_true_labels = y_oot.head(5)

# Fazer predições
predictions = loaded_pipeline.predict(sample_data)
probabilities = loaded_pipeline.predict_proba(sample_data)[:, 1]

# Exibir resultados
inference_results = pd.DataFrame({
    'True Label': sample_true_labels.values,
    'Predicted': predictions,
    'Probability': probabilities
})

print("\n📊 Resultados da Inferência:")
display(inference_results)

print("\n✅ Inferência concluída com sucesso!")
print("\n" + "="*80)
print("PIPELINE COMPLETO FINALIZADO")
print("="*80)
print("\nArquivos gerados:")
print(f"   1. Modelo: {model_path}")
print(f"   2. Metadados: {metadata_path}")
print(f"   3. Resultados: {results_path}")
print(f"\n🎯 Este notebook substitui completamente:")
print(f"   ❌ 07_transformacoes.ipynb")
print(f"   ❌ 09_transformacoes_das_variaveis.ipynb")
print(f"   ❌ 10_treinamento_modelo.ipynb")